In [2]:
%pip install -U massive pandas python-dateutil

Note: you may need to restart the kernel to use updated packages.


In [3]:
from getpass import getpass
from datetime import date 
import time
from dateutil.relativedelta import relativedelta
import pandas as pd
from massive import RESTClient
import os

c:\Users\amrit\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\amrit\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [4]:
# ---------- ENTER API KEY SAFELY ----------
API_KEY = getpass("Enter your Massive API key: ")  # <-- type key here (input hidden)
# -----------------------------------------

In [5]:
TICKERS = [
    "NOC", "RTX", "JNJ", "LMT", "NSRGF", "DUK", "CNI",
    "RIVN", "PLUG", "COIN", "U", "PM", "XOM", "PLTR", "META", "GLNCY"
]

In [6]:
client = RESTClient(api_key=API_KEY)

end = date.today()
start = end - relativedelta(years=3)

In [7]:
def fetch_daily_closes(ticker, start_date, end_date, max_tries=8):
    for attempt in range(max_tries):
        try:
            rows = []
            for bar in client.list_aggs(
                ticker=ticker,
                multiplier=1,
                timespan="day",
                from_=start_date.isoformat(),
                to=end_date.isoformat(),
                limit=50000,
            ):
                rows.append((pd.to_datetime(bar.timestamp, unit="ms"), float(bar.close)))

            if not rows:
                raise ValueError(f"No data returned for {ticker}")

            return (
                pd.Series(
                    data=[c for _, c in rows],
                    index=[ts for ts, _ in rows],
                    name=ticker,
                )
                .sort_index()
            )

        except Exception as e:
            if "429" in str(e):
                sleep_s = min(60, 5 * (2 ** attempt))
                print(f"429 for {ticker}. Sleeping {sleep_s}s then retrying...")
                time.sleep(sleep_s)
                continue
            raise

    raise RuntimeError(f"Failed for {ticker} after {max_tries} retries due to repeated 429s.")


In [8]:
price_series = {}

for t in TICKERS:
    print(f"Fetching {t}...")
    price_series[t] = fetch_daily_closes(t, start, end)
    time.sleep(3)  # spacing between tickers

prices = pd.DataFrame(price_series)
prices.head()


Fetching NOC...
Fetching RTX...
Fetching JNJ...
Fetching LMT...
Fetching NSRGF...
Fetching DUK...
429 for DUK. Sleeping 5s then retrying...
429 for DUK. Sleeping 10s then retrying...
429 for DUK. Sleeping 20s then retrying...
Fetching CNI...
Fetching RIVN...
Fetching PLUG...
Fetching COIN...
Fetching U...
429 for U. Sleeping 5s then retrying...
429 for U. Sleeping 10s then retrying...
429 for U. Sleeping 20s then retrying...
429 for U. Sleeping 40s then retrying...
Fetching PM...
Fetching XOM...
Fetching PLTR...
Fetching META...
Fetching GLNCY...
429 for GLNCY. Sleeping 5s then retrying...


,NOC,RTX,JNJ,LMT,NSRGF,DUK,CNI,RIVN,PLUG,COIN,U,PM,XOM,PLTR,META,GLNCY
2024-03-25 04:00:00,469.32,95.63,155.22,446.31,105.194,94.84,129.78,10.65,3.33,279.71,27.20,91.15,114.65,24.51,503.02,10.60
2024-03-26 04:00:00,469.91,96.06,155.77,445.99,105.154,93.67,130.17,10.52,3.22,266.81,26.70,90.38,113.79,24.89,495.89,10.63
2024-03-27 04:00:00,477.36,97.45,157.96,456.78,105.458,96.09,131.65,10.99,3.43,256.70,26.99,92.23,114.97,24.51,493.86,10.89
2024-03-28 04:00:00,478.66,97.53,158.19,454.87,106.180,96.71,131.71,10.95,3.44,265.12,26.70,91.62,116.24,23.01,485.58,10.95
2024-04-01 04:00:00,471.35,97.76,157.78,452.79,105.692,96.07,131.52,11.09,3.44,252.11,26.61,91.44,116.99,22.86,491.35,10.98


In [9]:
# returns = prices.pct_change().dropna(how="all")

# daily_var = returns.var()
# ann_vol = returns.std() * (252 ** 0.5)

# risk_bucket = pd.qcut(
#     ann_vol.rank(method="first"),
#     q=3,
#     labels=["Safe", "Moderate", "Volatile"]
# )

# risk_table = (
#     pd.DataFrame({
#         "ticker": ann_vol.index,
#         "daily_return_variance": daily_var.values,
#         "annualized_volatility": ann_vol.values,
#         "risk_category": risk_bucket.values,
#     })
#     .sort_values("annualized_volatility")
#     .reset_index(drop=True)
# )

# risk_table

In [10]:
# risk_table.to_csv("risk_profiles_3y_quantiles.csv", index=False)

In [12]:
# --------------------------------------------------
# 1) Clean index so date matching works properly
# --------------------------------------------------
prices.index = pd.to_datetime(prices.index).normalize()
prices = prices.sort_index()

# --------------------------------------------------
# 2) Compute returns and BINARY risk categories only
# --------------------------------------------------
returns = prices.pct_change().dropna(how="all")

daily_var = returns.var()
ann_vol = returns.std() * (252 ** 0.5)

# 2 groups only: Safe vs Risky
risk_bucket = pd.qcut(
    ann_vol.rank(method="first"),
    q=2,
    labels=["Safe", "Risky"]
)

risk_table = (
    pd.DataFrame({
        "ticker": ann_vol.index,
        "daily_return_variance": daily_var.values,
        "annualized_volatility": ann_vol.values,
        "risk_category": risk_bucket.values
    })
    .sort_values("annualized_volatility")
    .reset_index(drop=True)
)

print("BINARY RISK TABLE")
display(risk_table)

# --------------------------------------------------
# 3) Helper: get latest available close on or before target date
# --------------------------------------------------
def get_last_valid_price(series, target_date):
    target_date = pd.Timestamp(target_date).normalize()
    s = series.dropna().sort_index()
    s = s[s.index <= target_date]
    
    if s.empty:
        return pd.NaT, np.nan
    
    return s.index[-1], float(s.iloc[-1])

# --------------------------------------------------
# 4) Set your investment window
#    Change the YEAR here if needed
# --------------------------------------------------
entry_date = "2026-02-19"
exit_date  = "2026-03-19"

# Example: assume investor puts the same amount into each stock
investment_amount = 100.0

# --------------------------------------------------
# 5) Compute what that investment becomes after 1 month
# --------------------------------------------------
investment_rows = []

for ticker in TICKERS:
    entry_used, entry_price = get_last_valid_price(prices[ticker], entry_date)
    exit_used, exit_price = get_last_valid_price(prices[ticker], exit_date)

    if pd.isna(entry_price) or pd.isna(exit_price):
        continue

    shares_bought = investment_amount / entry_price
    final_value = shares_bought * exit_price
    profit_loss = final_value - investment_amount
    one_month_return = profit_loss / investment_amount

    investment_rows.append({
        "ticker": ticker,
        "entry_date_used": entry_used.date(),
        "entry_price": entry_price,
        "exit_date_used": exit_used.date(),
        "exit_price": exit_price,
        "investment_amount": investment_amount,
        "shares_bought": shares_bought,
        "final_value": final_value,
        "profit_loss": profit_loss,
        "one_month_return": one_month_return,
        "outcome": "Profit" if profit_loss > 0 else "Loss"
    })

investment_table = pd.DataFrame(investment_rows)

print("1-MONTH INVESTMENT RESULTS")
display(investment_table.sort_values("one_month_return"))

# --------------------------------------------------
# 6) Merge risk + 1-month outcome into one final table
# --------------------------------------------------
final_table = (
    risk_table.merge(investment_table, on="ticker", how="left")
    .sort_values(["risk_category", "annualized_volatility"])
    .reset_index(drop=True)
)

print("FINAL TABLE: RISK + 1-MONTH OUTCOME")
display(final_table)

# --------------------------------------------------
# 7) Quick summary checks
# --------------------------------------------------
print("\nCounts by risk category:")
print(final_table["risk_category"].value_counts())

print("\nCounts by outcome:")
print(final_table["outcome"].value_counts())

print("\nRisk x Outcome table:")
print(pd.crosstab(final_table["risk_category"], final_table["outcome"]))

# Optional: save
final_table.to_csv("risk_and_1m_outcomes.csv", index=False)

BINARY RISK TABLE


,ticker,daily_return_variance,annualized_volatility,risk_category
0,DUK,0.000108,0.165087,Safe
1,JNJ,0.000129,0.180146,Safe
2,CNI,0.000187,0.216902,Safe
3,XOM,0.000204,0.226813,Safe
4,RTX,0.000230,0.240985,Safe
5,LMT,0.000233,0.242181,Safe
6,PM,0.000245,0.248620,Safe
7,NSRGF,0.000246,0.249072,Safe
8,NOC,0.000249,0.250582,Risky
9,GLNCY,0.000479,0.347269,Risky


1-MONTH INVESTMENT RESULTS


,ticker,entry_date_used,entry_price,exit_date_used,exit_price,investment_amount,shares_bought,final_value,profit_loss,one_month_return,outcome
11,PM,2026-02-19,183.50,2026-03-19,163.370,100.0,0.544959,89.029973,-10.970027,-0.109700,Loss
6,CNI,2026-02-19,109.43,2026-03-19,99.100,100.0,0.913826,90.560175,-9.439825,-0.094398,Loss
4,NSRGF,2026-02-19,105.55,2026-03-19,96.400,100.0,0.947418,91.331123,-8.668877,-0.086689,Loss
14,META,2026-02-19,644.78,2026-03-19,606.700,100.0,0.155092,94.094110,-5.905890,-0.059059,Loss
3,LMT,2026-02-19,666.51,2026-03-19,637.510,100.0,0.150035,95.648978,-4.351022,-0.043510,Loss
2,JNJ,2026-02-19,246.91,2026-03-19,237.600,100.0,0.405006,96.229395,-3.770605,-0.037706,Loss
0,NOC,2026-02-19,736.87,2026-03-19,714.150,100.0,0.135709,96.916688,-3.083312,-0.030833,Loss
1,RTX,2026-02-19,205.41,2026-03-19,200.730,100.0,0.486831,97.721630,-2.278370,-0.022784,Loss
15,GLNCY,2026-02-19,13.65,2026-03-19,13.875,100.0,7.326007,101.648352,1.648352,0.016484,Profit
5,DUK,2026-02-19,126.37,2026-03-19,129.740,100.0,0.791327,102.666772,2.666772,0.026668,Profit


FINAL TABLE: RISK + 1-MONTH OUTCOME


,ticker,daily_return_variance,annualized_volatility,risk_category,entry_date_used,entry_price,exit_date_used,exit_price,investment_amount,shares_bought,final_value,profit_loss,one_month_return,outcome
0,DUK,0.000108,0.165087,Safe,2026-02-19,126.37,2026-03-19,129.740,100.0,0.791327,102.666772,2.666772,0.026668,Profit
1,JNJ,0.000129,0.180146,Safe,2026-02-19,246.91,2026-03-19,237.600,100.0,0.405006,96.229395,-3.770605,-0.037706,Loss
2,CNI,0.000187,0.216902,Safe,2026-02-19,109.43,2026-03-19,99.100,100.0,0.913826,90.560175,-9.439825,-0.094398,Loss
3,XOM,0.000204,0.226813,Safe,2026-02-19,150.97,2026-03-19,158.160,100.0,0.662383,104.762536,4.762536,0.047625,Profit
4,RTX,0.000230,0.240985,Safe,2026-02-19,205.41,2026-03-19,200.730,100.0,0.486831,97.721630,-2.278370,-0.022784,Loss
5,LMT,0.000233,0.242181,Safe,2026-02-19,666.51,2026-03-19,637.510,100.0,0.150035,95.648978,-4.351022,-0.043510,Loss
6,PM,0.000245,0.248620,Safe,2026-02-19,183.50,2026-03-19,163.370,100.0,0.544959,89.029973,-10.970027,-0.109700,Loss
7,NSRGF,0.000246,0.249072,Safe,2026-02-19,105.55,2026-03-19,96.400,100.0,0.947418,91.331123,-8.668877,-0.086689,Loss
8,NOC,0.000249,0.250582,Risky,2026-02-19,736.87,2026-03-19,714.150,100.0,0.135709,96.916688,-3.083312,-0.030833,Loss
9,GLNCY,0.000479,0.347269,Risky,2026-02-19,13.65,2026-03-19,13.875,100.0,7.326007,101.648352,1.648352,0.016484,Profit



Counts by risk category:
risk_category
Safe     8
Risky    8
Name: count, dtype: int64

Counts by outcome:
outcome
Profit    8
Loss      8
Name: count, dtype: int64

Risk x Outcome table:
outcome        Loss  Profit
risk_category              
Safe              6       2
Risky             2       6


In [13]:
# --------------------------------------------------
# 8) Add CONTEXT columns 
# --------------------------------------------------

# Risk window (based on your earlier code)
risk_start_date = start
risk_end_date = end

# Add readable metadata columns (row-level clarity)
final_table["risk_window"] = f"{risk_start_date} to {risk_end_date} (3Y daily volatility)"
final_table["investment_window_requested"] = f"{entry_date} to {exit_date} (1M horizon)"

# Rename for clarity
final_table = final_table.rename(columns={
    "entry_date_used": "actual_entry_date_used",
    "exit_date_used": "actual_exit_date_used"
})

# Reorder columns for readability
final_table = final_table[
    [
        "ticker",
        "risk_category",
        "annualized_volatility",
        "daily_return_variance",
        
        "risk_window",
        
        "investment_window_requested",
        "actual_entry_date_used",
        "entry_price",
        "actual_exit_date_used",
        "exit_price",
        
        "investment_amount",
        "shares_bought",
        "final_value",
        "profit_loss",
        "one_month_return",
        "outcome"
    ]
]

# --------------------------------------------------
# 9) SAVE CLEAN DATASET 
# --------------------------------------------------
final_table.to_csv("market_mindset_final_dataset.csv", index=False)
print("Saved: market_mindset_final_dataset.csv")


# --------------------------------------------------
# 10) ADD METADATA ROW 
# --------------------------------------------------

# Create metadata row with SAME columns
metadata_row = {col: "" for col in final_table.columns}

metadata_row["ticker"] = "METADATA"
metadata_row["risk_category"] = f"Risk window: {risk_start_date} to {risk_end_date} (3 years)"
metadata_row["annualized_volatility"] = f"Investment window: {entry_date} to {exit_date}"
metadata_row["daily_return_variance"] = "Volatility = std(returns) * sqrt(252)"

metadata_df = pd.DataFrame([metadata_row])

# Combine metadata + actual data
final_with_metadata = pd.concat([metadata_df, final_table], ignore_index=True)

# Save
final_with_metadata.to_csv("market_mindset_final_dataset_with_metadata.csv", index=False)

print("Saved: market_mindset_final_dataset_with_metadata.csv")

Saved: market_mindset_final_dataset.csv
Saved: market_mindset_final_dataset_with_metadata.csv


In [14]:
final_table[["ticker", "risk_category", "outcome", "one_month_return"]]

,ticker,risk_category,outcome,one_month_return
0,DUK,Safe,Profit,0.026668
1,JNJ,Safe,Loss,-0.037706
2,CNI,Safe,Loss,-0.094398
3,XOM,Safe,Profit,0.047625
4,RTX,Safe,Loss,-0.022784
5,LMT,Safe,Loss,-0.043510
6,PM,Safe,Loss,-0.109700
7,NSRGF,Safe,Loss,-0.086689
8,NOC,Risky,Loss,-0.030833
9,GLNCY,Risky,Profit,0.016484


In [15]:
# --------------------------------------------------
#    Historical metrics anchored on entry_date
#    Future outcome anchored on entry_date -> exit_date
# --------------------------------------------------

from dateutil.relativedelta import relativedelta

TICKERS = [
    "NOC", "RTX", "JNJ", "LMT", "NSRGF", "DUK", "CNI",
    "RIVN", "PLUG", "COIN", "U", "PM", "XOM", "PLTR", "META", "GLNCY"
]

entry_date = pd.Timestamp("2026-02-19").normalize()
exit_date  = pd.Timestamp("2026-03-19").normalize()

def pct_return(start_price, end_price):
    if pd.isna(start_price) or pd.isna(end_price) or start_price == 0:
        return np.nan
    return ((end_price / start_price) - 1) * 100

def format_return(x):
    if pd.isna(x):
        return np.nan
    return f"{x:+.2f}%"

def get_n_trading_day_return(series, anchor_date, n_days):
    s = series.dropna().sort_index()
    s.index = pd.to_datetime(s.index).normalize()

    anchor_used, anchor_price = get_last_valid_price(s, anchor_date)
    if pd.isna(anchor_price):
        return np.nan

    s_upto_anchor = s[s.index <= pd.Timestamp(anchor_used).normalize()]
    if len(s_upto_anchor) <= n_days:
        return np.nan

    start_price = float(s_upto_anchor.iloc[-(n_days + 1)])
    return pct_return(start_price, anchor_price)

def get_calendar_window_return(series, anchor_date, months_back=None, ytd=False):
    s = series.dropna().sort_index()
    s.index = pd.to_datetime(s.index).normalize()

    anchor_used, anchor_price = get_last_valid_price(s, anchor_date)
    if pd.isna(anchor_price):
        return np.nan

    if ytd:
        target_start = pd.Timestamp(year=pd.Timestamp(anchor_used).year, month=1, day=1)
    else:
        target_start = pd.Timestamp(anchor_used) - relativedelta(months=months_back)

    _, start_price = get_last_valid_price(s, target_start)
    return pct_return(start_price, anchor_price)

company_metrics_rows = []

for ticker in TICKERS:
    if ticker not in prices.columns:
        continue

    series = prices[ticker].dropna().sort_index()
    if series.empty:
        continue

    # Future realized return: Feb 19 -> Mar 19
    entry_used, entry_price = get_last_valid_price(series, entry_date)
    exit_used, exit_price   = get_last_valid_price(series, exit_date)

    return_percent = pct_return(entry_price, exit_price)

    # Historical metrics available as of Feb 19
    ret_5d  = get_n_trading_day_return(series, entry_date, 5)
    ret_10d = get_n_trading_day_return(series, entry_date, 10)
    ret_1m  = get_calendar_window_return(series, entry_date, months_back=1)
    ret_3m  = get_calendar_window_return(series, entry_date, months_back=3)
    ret_6m  = get_calendar_window_return(series, entry_date, months_back=6)
    ret_ytd = get_calendar_window_return(series, entry_date, ytd=True)

    company_metrics_rows.append({
        "ticker": ticker,
        "entry_date_used": entry_used.date() if pd.notna(entry_used) else pd.NaT,
        "exit_date_used": exit_used.date() if pd.notna(exit_used) else pd.NaT,
        "return_percent": round(return_percent, 2) if pd.notna(return_percent) else np.nan,
        "performance_metrics": {
            "5-day": format_return(ret_5d),
            "10-day": format_return(ret_10d),
            "1-month": format_return(ret_1m),
            "3-month": format_return(ret_3m),
            "6-month": format_return(ret_6m),
            "YTD": format_return(ret_ytd)
        }
    })

company_metrics_rows[:3]

[{'ticker': 'NOC',
  'entry_date_used': datetime.date(2026, 2, 19),
  'exit_date_used': datetime.date(2026, 3, 19),
  'return_percent': -3.08,
  'performance_metrics': {'5-day': '+8.55%',
   '10-day': '+6.83%',
   '1-month': '+10.49%',
   '3-month': '+30.42%',
   '6-month': '+25.70%',
   'YTD': '+29.23%'}},
 {'ticker': 'RTX',
  'entry_date_used': datetime.date(2026, 2, 19),
  'exit_date_used': datetime.date(2026, 3, 19),
  'return_percent': -2.28,
  'performance_metrics': {'5-day': '+4.53%',
   '10-day': '+4.41%',
   '1-month': '+1.73%',
   '3-month': '+18.21%',
   '6-month': '+33.68%',
   'YTD': '+12.00%'}},
 {'ticker': 'JNJ',
  'entry_date_used': datetime.date(2026, 2, 19),
  'exit_date_used': datetime.date(2026, 3, 19),
  'return_percent': -3.77,
  'performance_metrics': {'5-day': '+2.51%',
   '10-day': '+5.31%',
   '1-month': '+12.92%',
   '3-month': '+21.92%',
   '6-month': '+38.87%',
   'YTD': '+19.31%'}}]

In [16]:
company_metrics_table = pd.DataFrame([
    {
        "ticker": row["ticker"],
        "entry_date_used": row["entry_date_used"],
        "exit_date_used": row["exit_date_used"],
        "return_percent": row["return_percent"],
        "5-day": row["performance_metrics"]["5-day"],
        "10-day": row["performance_metrics"]["10-day"],
        "1-month": row["performance_metrics"]["1-month"],
        "3-month": row["performance_metrics"]["3-month"],
        "6-month": row["performance_metrics"]["6-month"],
        "YTD": row["performance_metrics"]["YTD"],
    }
    for row in company_metrics_rows
])

display(company_metrics_table)
company_metrics_table.to_csv("company_price_metrics.csv", index=False)

,ticker,entry_date_used,exit_date_used,return_percent,5-day,10-day,1-month,3-month,6-month,YTD
0,NOC,2026-02-19,2026-03-19,-3.08,+8.55%,+6.83%,+10.49%,+30.42%,+25.70%,+29.23%
1,RTX,2026-02-19,2026-03-19,-2.28,+4.53%,+4.41%,+1.73%,+18.21%,+33.68%,+12.00%
2,JNJ,2026-02-19,2026-03-19,-3.77,+2.51%,+5.31%,+12.92%,+21.92%,+38.87%,+19.31%
3,LMT,2026-02-19,2026-03-19,-4.35,+6.01%,+10.58%,+14.44%,+41.84%,+51.10%,+37.80%
4,NSRGF,2026-02-19,2026-03-19,-8.67,+4.52%,+5.89%,+12.76%,+7.66%,+17.30%,+5.93%
5,DUK,2026-02-19,2026-03-19,2.67,+0.93%,+3.40%,+6.00%,+3.39%,+2.09%,+7.82%
6,CNI,2026-02-19,2026-03-19,-9.44,+2.93%,+10.52%,+9.31%,+16.96%,+15.98%,+10.70%
7,RIVN,2026-02-19,2026-03-19,3.40,+5.62%,+8.49%,-6.48%,+5.91%,+27.27%,-20.90%
8,PLUG,2026-02-19,2026-03-19,25.65,-2.55%,-6.83%,-19.07%,+0.53%,+20.89%,-3.05%
9,COIN,2026-02-19,2026-03-19,22.28,+8.32%,-1.59%,-31.19%,-35.50%,-45.07%,-26.62%


In [17]:
# ==========================================================
# Build grounded week_analysis and month_analysis
# from Massive daily OHLCV bars
#
# WHAT THIS CODE DOES
# -------------------
# 1) Safely asks for your Massive API key
# 2) Downloads daily OHLCV bars for each ticker
# 3) Creates short rolling windows around your entry date
# 4) Computes transparent numeric features
# 5) Turns those features into explicit rule-based labels
# 6) Converts labels into short, realistic English summaries
#
# IMPORTANT DESIGN CHOICE
# -----------------------
# These analyses are anchored on entry_date (2026-02-19),
# because that is the information date participants can see.
# We DO NOT use future bars after that date for these summaries.
# ==========================================================

In [18]:
# anchor date
entry_date = pd.Timestamp("2026-02-19").normalize()

In [19]:
# ---------------------------------------------------------
# Helper: fetch daily OHLCV bars for one ticker
#
# Why we need this:
# Your existing `prices` object contains closes only.
# For richer text like:
# - momentum
# - pullback
# - wide trading range
# - mixed volume
# we need OHLCV, especially high/low/volume.
# ---------------------------------------------------------

def fetch_daily_ohlcv(ticker, start_date, end_date, max_tries=6, base_sleep=2.5):
    """
    Fetch daily OHLCV bars from Massive for one ticker.

    This version is designed to handle rate limits more safely by:
    1. retrying multiple times
    2. waiting longer after each failure
    3. returning an empty DataFrame if all retries fail
    """
    empty_df = pd.DataFrame(columns=["open", "high", "low", "close", "volume"])

    for attempt in range(1, max_tries + 1):
        try:
            rows = []

            bars = client.list_aggs(
                ticker=ticker,
                multiplier=1,
                timespan="day",
                from_=str(pd.Timestamp(start_date).date()),
                to=str(pd.Timestamp(end_date).date()),
                adjusted=True,
                sort="asc",
                limit=5000
            )

            for bar in bars:
                rows.append({
                    "date": pd.to_datetime(bar.timestamp, unit="ms").normalize(),
                    "open": float(bar.open),
                    "high": float(bar.high),
                    "low": float(bar.low),
                    "close": float(bar.close),
                    "volume": float(bar.volume)
                })

            df = pd.DataFrame(rows)

            if df.empty:
                print(f"No data returned for {ticker}")
                return empty_df

            df = df.sort_values("date").drop_duplicates("date")
            df = df.set_index("date")

            # small success pause too, to avoid hammering the API
            time.sleep(base_sleep)
            return df

        except Exception as e:
            wait_time = base_sleep * (2 ** (attempt - 1))
            print(f"[Attempt {attempt}/{max_tries}] Error fetching {ticker}: {e}")
            
            if attempt < max_tries:
                print(f"Waiting {wait_time:.1f} seconds before retrying {ticker}...")
                time.sleep(wait_time)
            else:
                print(f"Failed completely for {ticker}")
                return empty_df

In [20]:
# ---------------------------------------------------------
# Pull enough historical data to support:
# - 5 trading day summary
# - ~21 trading day month summary
# - a little extra buffer
# ---------------------------------------------------------
history_start = entry_date - pd.DateOffset(months=8)
history_end   = entry_date

ohlcv_data = {}

for i, ticker in enumerate(TICKERS, start=1):
    print(f"Fetching {i}/{len(TICKERS)}: {ticker}")
    df = fetch_daily_ohlcv(ticker, history_start, history_end, max_tries=6, base_sleep=2.5)

    if df is not None and not df.empty:
        df.index = pd.to_datetime(df.index).normalize()
        df = df.sort_index()
        ohlcv_data[ticker] = df
    else:
        print(f"No OHLCV data stored for {ticker}")

print(f"\nFetched OHLCV data for {len(ohlcv_data)} tickers.")

Fetching 1/16: NOC
Fetching 2/16: RTX
Fetching 3/16: JNJ
Fetching 4/16: LMT
Fetching 5/16: NSRGF
Fetching 6/16: DUK
[Attempt 1/6] Error fetching DUK: HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/DUK/range/1/day/2025-06-19/2026-02-19?adjusted=true&sort=asc&limit=5000 (Caused by ResponseError('too many 429 error responses'))
Waiting 2.5 seconds before retrying DUK...
[Attempt 2/6] Error fetching DUK: HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/DUK/range/1/day/2025-06-19/2026-02-19?adjusted=true&sort=asc&limit=5000 (Caused by ResponseError('too many 429 error responses'))
Waiting 5.0 seconds before retrying DUK...
Fetching 7/16: CNI
Fetching 8/16: RIVN
Fetching 9/16: PLUG
Fetching 10/16: COIN
Fetching 11/16: U
[Attempt 1/6] Error fetching U: HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/U/range/1/day/2025-06-19/2026-02-1

In [21]:
# ---------------------------------------------------------
# This cell:
# 1) gets the trailing week and month windows
# 2) computes grounded OHLCV features
# 3) assigns explicit labels
# 4) writes week_analysis and month_analysis


# ---------------------------------------------------------
# Helper: get trailing window ending on anchor date
# ---------------------------------------------------------
def get_trailing_window(df, anchor_date, n_days):
    """
    Return the last n_days trading rows on or before anchor_date.

    Example:
    - n_days = 5  -> roughly your 'week' window
    - n_days = 21 -> roughly your 'month' window
    """
    anchor_date = pd.Timestamp(anchor_date).normalize()

    d = df[df.index <= anchor_date].sort_index()

    if len(d) < n_days:
        return pd.DataFrame(columns=df.columns)

    return d.iloc[-n_days:].copy()


# ---------------------------------------------------------
# Feature engineering for one window
# ---------------------------------------------------------
def compute_window_features(window_df):
    """
    Compute transparent, auditable numeric features from one OHLCV window.

    Returns:
    - start_close
    - end_close
    - peak_close
    - cumulative return over the window
    - total price range over the window
    - pullback from the peak close
    - share of up days
    - daily volatility proxy
    - volume trend
    - volume variability
    """
    if window_df.empty or len(window_df) < 2:
        return None

    closes = window_df["close"].astype(float)
    highs = window_df["high"].astype(float)
    lows = window_df["low"].astype(float)
    volumes = window_df["volume"].astype(float)

    start_close = closes.iloc[0]
    end_close = closes.iloc[-1]
    peak_close = closes.max()

    # total return over the window
    cum_return_pct = ((end_close / start_close) - 1.0) * 100

    # total high-low range over the window, scaled by start price
    window_range_pct = ((highs.max() - lows.min()) / start_close) * 100

    # how far the stock ended below its highest close in the window
    pullback_from_peak_pct = ((end_close / peak_close) - 1.0) * 100

    # share of sessions that were positive relative to previous close
    daily_diff = closes.diff()
    up_day_share = (daily_diff > 0).mean()

    # standard deviation of daily returns, in percent
    daily_returns = closes.pct_change().dropna()
    daily_vol_pct = daily_returns.std() * 100

    # compare average volume in first third vs last third of window
    n = len(volumes)
    split = max(2, n // 3)

    first_avg_vol = volumes.iloc[:split].mean()
    last_avg_vol = volumes.iloc[-split:].mean()

    if first_avg_vol == 0 or pd.isna(first_avg_vol):
        volume_trend_pct = np.nan
    else:
        volume_trend_pct = ((last_avg_vol / first_avg_vol) - 1.0) * 100

    # coefficient of variation for volume
    if volumes.mean() == 0 or pd.isna(volumes.mean()):
        volume_cv = np.nan
    else:
        volume_cv = volumes.std() / volumes.mean()

    return {
        "start_close": round(float(start_close), 2),
        "end_close": round(float(end_close), 2),
        "peak_close": round(float(peak_close), 2),
        "cum_return_pct": round(float(cum_return_pct), 2),
        "window_range_pct": round(float(window_range_pct), 2),
        "pullback_from_peak_pct": round(float(pullback_from_peak_pct), 2),
        "up_day_share": round(float(up_day_share), 3),
        "daily_vol_pct": round(float(daily_vol_pct), 2),
        "volume_trend_pct": round(float(volume_trend_pct), 2) if pd.notna(volume_trend_pct) else np.nan,
        "volume_cv": round(float(volume_cv), 3) if pd.notna(volume_cv) else np.nan
    }


# ---------------------------------------------------------
# Convert numeric features into explicit labels
# ---------------------------------------------------------
def label_features(features):
    """
    Turn grounded numeric features into discrete labels.
    """
    if features is None:
        return None

    r = features["cum_return_pct"]

    if r >= 5:
        trend = "strong uptrend"
    elif r >= 1:
        trend = "modest uptrend"
    elif r <= -5:
        trend = "strong downtrend"
    elif r <= -1:
        trend = "modest downtrend"
    else:
        trend = "sideways"

    range_pct = features["window_range_pct"]
    vol_pct = features["daily_vol_pct"]

    if range_pct >= 15 or vol_pct >= 4:
        volatility = "high volatility"
    elif range_pct >= 7 or vol_pct >= 2:
        volatility = "moderate volatility"
    else:
        volatility = "low volatility"

    pb = features["pullback_from_peak_pct"]

    if pb <= -5:
        pullback = "clear pullback"
    elif pb <= -2:
        pullback = "mild pullback"
    else:
        pullback = "little pullback"

    up_share = features["up_day_share"]

    if up_share >= 0.70:
        breadth = "mostly positive sessions"
    elif up_share >= 0.55:
        breadth = "more up than down sessions"
    elif up_share <= 0.30:
        breadth = "mostly negative sessions"
    elif up_share <= 0.45:
        breadth = "more down than up sessions"
    else:
        breadth = "mixed session pattern"

    vt = features["volume_trend_pct"]
    vcv = features["volume_cv"]

    if pd.isna(vt) or pd.isna(vcv):
        volume_label = "volume pattern unclear"
    else:
        if vt >= 15:
            volume_direction = "rising volume"
        elif vt <= -15:
            volume_direction = "falling volume"
        else:
            volume_direction = "stable volume"

        if vcv >= 0.35:
            volume_consistency = "mixed volume"
        else:
            volume_consistency = "fairly even volume"

        volume_label = f"{volume_direction} with {volume_consistency}"

    return {
        "trend": trend,
        "volatility": volatility,
        "pullback": pullback,
        "breadth": breadth,
        "volume_label": volume_label
    }


# ---------------------------------------------------------
# Turn labels + features into short text
# ---------------------------------------------------------
def write_analysis(features, labels, window_name="week"):
    """
    Create a short, grounded narrative from computed OHLCV features.

    Design principles:
    - Only describe things supported by price/volume data
    - Avoid unsupported claims like sentiment, conviction, panic, profit-taking
    - Sound more natural than a rigid template
    """
    if features is None or labels is None:
        return "Not enough historical data to generate a summary."

    start_close = features["start_close"]
    end_close = features["end_close"]
    peak_close = features["peak_close"]

    cum_return_pct = features["cum_return_pct"]
    window_range_pct = features["window_range_pct"]
    pullback_from_peak_pct = features["pullback_from_peak_pct"]

    trend = labels["trend"]
    volatility = labels["volatility"]
    pullback = labels["pullback"]
    breadth = labels["breadth"]
    volume_label = labels["volume_label"]

    # -----------------------------
    # Sentence 1: overall direction
    # -----------------------------
    if trend == "strong uptrend":
        s1 = (
            f"The stock moved higher over the {window_name}, climbing from about "
            f"{start_close:.2f} to {end_close:.2f}."
        )
    elif trend == "modest uptrend":
        s1 = (
            f"The stock posted a modest gain over the {window_name}, ending near "
            f"{end_close:.2f} after starting around {start_close:.2f}."
        )
    elif trend == "strong downtrend":
        s1 = (
            f"The stock fell sharply over the {window_name}, moving from about "
            f"{start_close:.2f} to {end_close:.2f}."
        )
    elif trend == "modest downtrend":
        s1 = (
            f"The stock drifted lower over the {window_name}, finishing near "
            f"{end_close:.2f} after opening the window around {start_close:.2f}."
        )
    else:
        s1 = (
            f"The stock was relatively flat over the {window_name}, ending near "
            f"{end_close:.2f} after starting around {start_close:.2f}."
        )

    # -----------------------------
    # Sentence 2: range + position vs peak
    # -----------------------------
    if pullback == "clear pullback":
        s2 = (
            f"It reached a recent high near {peak_close:.2f} but finished the "
            f"{window_name} meaningfully below that level."
        )
    elif pullback == "mild pullback":
        s2 = (
            f"It touched a recent high near {peak_close:.2f} and then eased back "
            f"slightly into the end of the {window_name}."
        )
    else:
        if end_close >= 0.98 * peak_close:
            s2 = (
                f"It finished close to its recent high near {peak_close:.2f}, "
                f"with little retreat late in the {window_name}."
            )
        else:
            s2 = (
                f"It remained below its recent high near {peak_close:.2f}, though "
                f"the pullback was limited by the end of the {window_name}."
            )

    # -----------------------------
    # Sentence 3: range / volatility / breadth / volume
    # -----------------------------
    if window_range_pct >= 15:
        range_phrase = "a wide trading range"
    elif window_range_pct >= 7:
        range_phrase = "a moderate trading range"
    else:
        range_phrase = "a relatively narrow trading range"

    if breadth == "mostly positive sessions":
        breadth_phrase = "most sessions closed higher than the previous day"
    elif breadth == "more up than down sessions":
        breadth_phrase = "up days slightly outnumbered down days"
    elif breadth == "mostly negative sessions":
        breadth_phrase = "most sessions closed lower than the previous day"
    elif breadth == "more down than up sessions":
        breadth_phrase = "down days slightly outnumbered up days"
    else:
        breadth_phrase = "up and down sessions were fairly mixed"

    s3 = (
        f"The {window_name} showed {range_phrase} with {volatility.lower()}, "
        f"{breadth_phrase}, and {volume_label}."
    )

    # -----------------------------
    # sentence 4:
    # -----------------------------
    if abs(cum_return_pct) >= 8:
        direction_word = "gain" if cum_return_pct > 0 else "decline"
        s4 = f"Overall, this amounted to a {abs(cum_return_pct):.2f}% {direction_word} over the {window_name}."
        return f"{s1} {s2} {s3} {s4}"

    return f"{s1} {s2} {s3}"

# ---------------------------------------------------------
# Build week and month analyses for every ticker
# ---------------------------------------------------------
analysis_rows = []

for ticker in TICKERS:
    if ticker not in ohlcv_data:
        continue

    df = ohlcv_data[ticker]

    # last 5 trading days up to entry_date
    week_df = get_trailing_window(df, entry_date, 5)

    # last ~21 trading days up to entry_date
    month_df = get_trailing_window(df, entry_date, 21)

    week_features = compute_window_features(week_df)
    month_features = compute_window_features(month_df)

    week_labels = label_features(week_features)
    month_labels = label_features(month_features)

    week_analysis = write_analysis(week_features, week_labels, window_name="week")
    month_analysis = write_analysis(month_features, month_labels, window_name="month")

    analysis_rows.append({
        "ticker": ticker,
        "week_features": week_features,
        "week_labels": week_labels,
        "week_analysis": week_analysis,
        "month_features": month_features,
        "month_labels": month_labels,
        "month_analysis": month_analysis
    })

analysis_table = pd.DataFrame(analysis_rows)

print("WEEK / MONTH ANALYSIS TABLE")
display(analysis_table[["ticker", "week_analysis", "month_analysis"]])

WEEK / MONTH ANALYSIS TABLE


,ticker,week_analysis,month_analysis
0,NOC,"The stock moved higher over the week, climbing...","The stock moved higher over the month, climbin..."
1,RTX,"The stock posted a modest gain over the week, ...","The stock posted a modest gain over the month,..."
2,JNJ,"The stock was relatively flat over the week, e...","The stock moved higher over the month, climbin..."
3,LMT,"The stock posted a modest gain over the week, ...","The stock moved higher over the month, climbin..."
4,NSRGF,"The stock posted a modest gain over the week, ...","The stock moved higher over the month, climbin..."
5,DUK,"The stock was relatively flat over the week, e...","The stock moved higher over the month, climbin..."
6,CNI,"The stock posted a modest gain over the week, ...","The stock moved higher over the month, climbin..."
7,RIVN,"The stock moved higher over the week, climbing...","The stock fell sharply over the month, moving ..."
8,PLUG,"The stock posted a modest gain over the week, ...","The stock fell sharply over the month, moving ..."
9,COIN,"The stock moved higher over the week, climbing...","The stock fell sharply over the month, moving ..."


In [22]:
for _, row in analysis_table.head(3).iterrows():
    print("Ticker:", row["ticker"])
    print("Week:", row["week_analysis"])
    print("Month:", row["month_analysis"])
    print("-" * 80)

Ticker: NOC
Week: The stock moved higher over the week, climbing from about 695.06 to 736.87. It finished close to its recent high near 736.87, with little retreat late in the week. The week showed a moderate trading range with moderate volatility, up days slightly outnumbered down days, and rising volume with fairly even volume.
Month: The stock moved higher over the month, climbing from about 664.16 to 736.87. It finished close to its recent high near 736.87, with little retreat late in the month. The month showed a wide trading range with high volatility, up days slightly outnumbered down days, and stable volume with fairly even volume. Overall, this amounted to a 10.95% gain over the month.
--------------------------------------------------------------------------------
Ticker: RTX
Week: The stock posted a modest gain over the week, ending near 205.41 after starting around 201.14. It finished close to its recent high near 205.41, with little retreat late in the week. The week showe

In [23]:
# ---------------------------------------------------------
# MERGE the new analyses into your existing company_metrics_rows
# ---------------------------------------------------------

analysis_lookup = analysis_table.set_index("ticker").to_dict(orient="index")

for row in company_metrics_rows:
    ticker = row["ticker"]

    if ticker in analysis_lookup:
        row["week_analysis"] = analysis_lookup[ticker]["week_analysis"]
        row["month_analysis"] = analysis_lookup[ticker]["month_analysis"]

        # keep structured audit info too
        row["week_features"] = analysis_lookup[ticker]["week_features"]
        row["month_features"] = analysis_lookup[ticker]["month_features"]
        row["week_labels"] = analysis_lookup[ticker]["week_labels"]
        row["month_labels"] = analysis_lookup[ticker]["month_labels"]
    else:
        row["week_analysis"] = "Not enough historical data to generate a summary."
        row["month_analysis"] = "Not enough historical data to generate a summary."
        row["week_features"] = None
        row["month_features"] = None
        row["week_labels"] = None
        row["month_labels"] = None

print("Merged analyses into company_metrics_rows.")
company_metrics_rows[:3]

Merged analyses into company_metrics_rows.


[{'ticker': 'NOC',
  'entry_date_used': datetime.date(2026, 2, 19),
  'exit_date_used': datetime.date(2026, 3, 19),
  'return_percent': -3.08,
  'performance_metrics': {'5-day': '+8.55%',
   '10-day': '+6.83%',
   '1-month': '+10.49%',
   '3-month': '+30.42%',
   '6-month': '+25.70%',
   'YTD': '+29.23%'},
  'week_analysis': 'The stock moved higher over the week, climbing from about 695.06 to 736.87. It finished close to its recent high near 736.87, with little retreat late in the week. The week showed a moderate trading range with moderate volatility, up days slightly outnumbered down days, and rising volume with fairly even volume.',
  'month_analysis': 'The stock moved higher over the month, climbing from about 664.16 to 736.87. It finished close to its recent high near 736.87, with little retreat late in the month. The month showed a wide trading range with high volatility, up days slightly outnumbered down days, and stable volume with fairly even volume. Overall, this amounted to 

In [24]:
%pip install -U openai

Note: you may need to restart the kernel to use updated packages.


In [42]:
# ---------------------------------------------------------
# OPENAI REWRITE STEP
# This keeps your rule-based analyses, and only rewrites them
# for smoother wording.
# ---------------------------------------------------------

from getpass import getpass
from openai import OpenAI

# Safer than hardcoding
OPENAI_API_KEY = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=OPENAI_API_KEY)

In [50]:
def rewrite_analysis(text):
    """
    Rewrite analysis to be concise, tight, and simple
    while preserving meaning.
    """
    prompt = f"""
Rewrite the following stock analysis.

Rules:
- Keep the meaning exactly the same
- Do NOT add any new claims
- Do NOT introduce sentiment, psychology, or causes
- Keep it grounded in observable price and volume behavior
- Use simple language
- Make it concise and tight
- Prefer 1-2 sentences
- Avoid repetition
- Avoid dramatic wording

Text:
{text}
"""

    response = client.responses.create(
        model="gpt-5.2",
        input=prompt
    )

    return response.output_text.strip()

In [51]:
print("ORIGINAL WEEK:")
print(company_metrics_rows[0]["week_analysis"])
print()

rewritten = rewrite_analysis(company_metrics_rows[0]["week_analysis"])

print("REWRITTEN WEEK:")
print(rewritten)

ORIGINAL WEEK:
The stock moved higher over the week, climbing from about 695.06 to 736.87. It finished close to its recent high near 736.87, with little retreat late in the week. The week showed a moderate trading range with moderate volatility, up days slightly outnumbered down days, and rising volume with fairly even volume.

REWRITTEN WEEK:
The stock rose over the week from about 695.06 to 736.87 and closed near its recent high around 736.87 with minimal pullback late in the week. Trading stayed in a moderate range with moderate volatility, up days slightly exceeded down days, and volume increased while remaining fairly even.


In [52]:
for row in company_metrics_rows:
    row["week_analysis_rewritten"] = rewrite_analysis(row["week_analysis"])
    row["month_analysis_rewritten"] = rewrite_analysis(row["month_analysis"])

print("Added rewritten analyses to all rows.")

Added rewritten analyses to all rows.


In [54]:
for row in company_metrics_rows[:3]:
    print("Ticker:", row["ticker"])
    print("ORIGINAL WEEK:", row["week_analysis"])
    print("REWRITTEN WEEK:", row["week_analysis_rewritten"])
    print("ORIGINAL MONTH:", row["month_analysis"])
    print("REWRITTEN MONTH:", row["month_analysis_rewritten"])
    print("-" * 100)

Ticker: NOC
ORIGINAL WEEK: The stock moved higher over the week, climbing from about 695.06 to 736.87. It finished close to its recent high near 736.87, with little retreat late in the week. The week showed a moderate trading range with moderate volatility, up days slightly outnumbered down days, and rising volume with fairly even volume.
REWRITTEN WEEK: The stock rose over the week from about 695.06 to 736.87 and ended near the recent high with little pullback. It traded in a moderate range with moderate volatility, up days slightly outnumbered down days, and volume increased while staying fairly even.
ORIGINAL MONTH: The stock moved higher over the month, climbing from about 664.16 to 736.87. It finished close to its recent high near 736.87, with little retreat late in the month. The month showed a wide trading range with high volatility, up days slightly outnumbered down days, and stable volume with fairly even volume. Overall, this amounted to a 10.95% gain over the month.
REWRITTE

In [55]:
final_rewritten_table = pd.DataFrame([
    {
        "ticker": row["ticker"],
        "entry_date_used": row["entry_date_used"],
        "exit_date_used": row["exit_date_used"],
        "return_percent": row["return_percent"],
        "performance_metrics": row["performance_metrics"],
        "week_analysis_original": row["week_analysis"],
        "week_analysis_rewritten": row["week_analysis_rewritten"],
        "month_analysis_original": row["month_analysis"],
        "month_analysis_rewritten": row["month_analysis_rewritten"],
    }
    for row in company_metrics_rows
])

print("FINAL REWRITTEN TABLE")
display(final_rewritten_table)

FINAL REWRITTEN TABLE


,ticker,entry_date_used,exit_date_used,return_percent,performance_metrics,week_analysis_original,week_analysis_rewritten,month_analysis_original,month_analysis_rewritten
0,NOC,2026-02-19,2026-03-19,-3.08,"{'5-day': '+8.55%', '10-day': '+6.83%', '1-mon...","The stock moved higher over the week, climbing...",The stock rose over the week from about 695.06...,"The stock moved higher over the month, climbin...","Over the month, the stock rose from about 664...."
1,RTX,2026-02-19,2026-03-19,-2.28,"{'5-day': '+4.53%', '10-day': '+4.41%', '1-mon...","The stock posted a modest gain over the week, ...","The stock rose modestly over the week, moving ...","The stock posted a modest gain over the month,...","The stock rose modestly over the month, moving..."
2,JNJ,2026-02-19,2026-03-19,-3.77,"{'5-day': '+2.51%', '10-day': '+5.31%', '1-mon...","The stock was relatively flat over the week, e...","The stock was mostly flat for the week, rising...","The stock moved higher over the month, climbin...",The stock rose over the month from about 218.0...
3,LMT,2026-02-19,2026-03-19,-4.35,"{'5-day': '+6.01%', '10-day': '+10.58%', '1-mo...","The stock posted a modest gain over the week, ...","The stock rose modestly for the week, moving f...","The stock moved higher over the month, climbin...","Over the month, the stock rose 13.69% from abo..."
4,NSRGF,2026-02-19,2026-03-19,-8.67,"{'5-day': '+4.52%', '10-day': '+5.89%', '1-mon...","The stock posted a modest gain over the week, ...","The stock rose modestly for the week, moving f...","The stock moved higher over the month, climbin...","Over the month, the stock rose 16.19%, moving ..."
5,DUK,2026-02-19,2026-03-19,2.67,"{'5-day': '+0.93%', '10-day': '+3.40%', '1-mon...","The stock was relatively flat over the week, e...","The stock was mostly flat for the week, ending...","The stock moved higher over the month, climbin...",The stock rose over the month from about 119.3...
6,CNI,2026-02-19,2026-03-19,-9.44,"{'5-day': '+2.93%', '10-day': '+10.52%', '1-mo...","The stock posted a modest gain over the week, ...","The stock rose modestly for the week, moving f...","The stock moved higher over the month, climbin...",The stock rose over the month from about 99.38...
7,RIVN,2026-02-19,2026-03-19,3.40,"{'5-day': '+5.62%', '10-day': '+8.49%', '1-mon...","The stock moved higher over the week, climbing...","The stock rose 11.36% for the week, moving fro...","The stock fell sharply over the month, moving ...","Over the month, the stock dropped from about 1..."
8,PLUG,2026-02-19,2026-03-19,25.65,"{'5-day': '-2.55%', '10-day': '-6.83%', '1-mon...","The stock posted a modest gain over the week, ...","The stock rose modestly for the week, moving f...","The stock fell sharply over the month, moving ...","Over the month, the stock dropped 13.96% from ..."
9,COIN,2026-02-19,2026-03-19,22.28,"{'5-day': '+8.32%', '10-day': '-1.59%', '1-mon...","The stock moved higher over the week, climbing...","The stock rose 17.61% for the week, moving fro...","The stock fell sharply over the month, moving ...","Over the month, the stock dropped 26.88%, fall..."


In [57]:
final_rewritten_table.to_csv("company_metrics_with_rewrites.csv", index=False)

In [25]:
from massive import RESTClient
massive_client = RESTClient(API_KEY)

In [26]:
ticker_to_pseudo = {
    "NOC": "P1",
    "LMT": "P2",
    "JNJ": "A1",
    "RTX": "A2",
    "DUK": "A3",
    "CNI": "A4",
    "PLUG": "A5",
    "COIN": "A6",
    "U": "A7",
    "PM": "A8",
    "XOM": "A9",
    "META": "A10",
    "NSRGF": "N1",
    "RIVN": "N2",
    "PLTR": "N3",
    "GLNCY": "N4",
}

# ---------------------------------------------------------
# Save path inside your repo
# Change this only if your notebook is not running
# from the repo root.
# ---------------------------------------------------------
output_dir = os.path.join("assets", "images")
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------------
# Helper: fetch daily OHLCV
# ---------------------------------------------------------
def fetch_daily_ohlcv(ticker, start_date, end_date, max_tries=5, base_sleep=2.5):
    empty_df = pd.DataFrame(columns=["Open", "High", "Low", "Close", "Volume"])

    for attempt in range(1, max_tries + 1):
        try:
            rows = []
            bars = client.list_aggs(
                ticker=ticker,
                multiplier=1,
                timespan="day",
                from_=str(pd.Timestamp(start_date).date()),
                to=str(pd.Timestamp(end_date).date()),
                adjusted=True,
                sort="asc",
                limit=5000
            )

            for bar in bars:
                rows.append({
                    "Date": pd.to_datetime(bar.timestamp, unit="ms").normalize(),
                    "Open": float(bar.open),
                    "High": float(bar.high),
                    "Low": float(bar.low),
                    "Close": float(bar.close),
                    "Volume": float(bar.volume)
                })

            df = pd.DataFrame(rows)
            if df.empty:
                return empty_df

            df = df.sort_values("Date").drop_duplicates("Date").set_index("Date")
            time.sleep(base_sleep)
            return df

        except Exception as e:
            wait_time = base_sleep * (2 ** (attempt - 1))
            print(f"[daily {ticker}] attempt {attempt}/{max_tries} failed: {e}")
            if attempt < max_tries:
                time.sleep(wait_time)
            else:
                return empty_df

# ---------------------------------------------------------
# Helper: fetch intraday OHLCV for one date
# 5-minute candles for the day on or before entry_date
# ---------------------------------------------------------
def fetch_intraday_ohlcv(ticker, target_date, multiplier=5, max_tries=5, base_sleep=2.5):
    empty_df = pd.DataFrame(columns=["Open", "High", "Low", "Close", "Volume"])

    target_date = pd.Timestamp(target_date).normalize()

    for attempt in range(1, max_tries + 1):
        try:
            rows = []
            bars = client.list_aggs(
                ticker=ticker,
                multiplier=multiplier,
                timespan="minute",
                from_=str(target_date.date()),
                to=str(target_date.date()),
                adjusted=True,
                sort="asc",
                limit=50000
            )

            for bar in bars:
                rows.append({
                    "DateTime": pd.to_datetime(bar.timestamp, unit="ms"),
                    "Open": float(bar.open),
                    "High": float(bar.high),
                    "Low": float(bar.low),
                    "Close": float(bar.close),
                    "Volume": float(bar.volume)
                })

            df = pd.DataFrame(rows)
            if df.empty:
                return empty_df

            df = df.sort_values("DateTime").drop_duplicates("DateTime").set_index("DateTime")
            time.sleep(base_sleep)
            return df

        except Exception as e:
            wait_time = base_sleep * (2 ** (attempt - 1))
            print(f"[intraday {ticker}] attempt {attempt}/{max_tries} failed: {e}")
            if attempt < max_tries:
                time.sleep(wait_time)
            else:
                return empty_df

# ---------------------------------------------------------
# Pull enough daily data for week/month windows
# ---------------------------------------------------------
history_start = entry_date - pd.DateOffset(months=3)
history_end = entry_date

availability_rows = []

for ticker, pseudo in ticker_to_pseudo.items():
    print(f"Checking {ticker} ({pseudo}) ...")

    daily_df = fetch_daily_ohlcv(ticker, history_start, history_end, max_tries=5, base_sleep=2.5)
    intraday_df = fetch_intraday_ohlcv(ticker, entry_date, multiplier=5, max_tries=5, base_sleep=2.5)

    week_ok = len(daily_df) >= 5
    month_ok = len(daily_df) >= 21
    intraday_ok = not intraday_df.empty

    availability_rows.append({
        "ticker": ticker,
        "pseudoname": pseudo,
        "intraday_available": intraday_ok,
        "daily_rows": len(daily_df),
        "week_available": week_ok,
        "month_available": month_ok
    })

availability_table = pd.DataFrame(availability_rows)
display(availability_table)

Checking NOC (P1) ...
Checking LMT (P2) ...
Checking JNJ (A1) ...
Checking RTX (A2) ...
Checking DUK (A3) ...
Checking CNI (A4) ...
[daily CNI] attempt 1/5 failed: HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/CNI/range/1/day/2025-11-19/2026-02-19?adjusted=true&sort=asc&limit=5000 (Caused by ResponseError('too many 429 error responses'))
[daily CNI] attempt 2/5 failed: HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/CNI/range/1/day/2025-11-19/2026-02-19?adjusted=true&sort=asc&limit=5000 (Caused by ResponseError('too many 429 error responses'))
[daily CNI] attempt 3/5 failed: HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/CNI/range/1/day/2025-11-19/2026-02-19?adjusted=true&sort=asc&limit=5000 (Caused by ResponseError('too many 429 error responses'))
[daily CNI] attempt 4/5 failed: HTTPSConnectionPool(host='api.massive.com', 

,ticker,pseudoname,intraday_available,daily_rows,week_available,month_available
0,NOC,P1,True,62,True,True
1,LMT,P2,True,62,True,True
2,JNJ,A1,True,62,True,True
3,RTX,A2,True,62,True,True
4,DUK,A3,True,62,True,True
5,CNI,A4,True,0,False,False
6,PLUG,A5,True,62,True,True
7,COIN,A6,True,62,True,True
8,U,A7,True,0,False,False
9,PM,A8,True,62,True,True


In [27]:
%pip install -U mplfinance

Note: you may need to restart the kernel to use updated packages.Collecting mplfinance
   ---------------------------------------- 0.0/75.0 kB ? eta -:--:--
   ---------------------------------------- 75.0/75.0 kB 4.0 MB/s eta 0:00:00



In [30]:
import mplfinance as mpf

In [33]:
# ---------------------------------------------------------
# Exact ticker -> pseudoname mapping
# ---------------------------------------------------------
ticker_to_pseudo = {
    "NOC": "P1",
    "LMT": "P2",
    "JNJ": "A1",
    "RTX": "A2",
    "DUK": "A3",
    "CNI": "A4",
    "PLUG": "A5",
    "COIN": "A6",
    "U": "A7",
    "PM": "A8",
    "XOM": "A9",
    "META": "A10",
    "NSRGF": "N1",
    "RIVN": "N2",
    "PLTR": "N3",
    "GLNCY": "N4",
}

In [35]:
# ---------------------------------------------------------
# Dates and output directory
# ---------------------------------------------------------
entry_date = pd.Timestamp("2026-02-19").normalize()
history_start = entry_date - pd.DateOffset(months=3)
history_end = entry_date

output_dir = os.path.join("assets", "images")
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------------
# Helper: fetch daily OHLCV using Massive
# ---------------------------------------------------------
def fetch_daily_ohlcv(ticker, start_date, end_date, max_tries=5, base_sleep=2.5):
    """
    Fetch daily OHLCV bars for one ticker.
    Returns a DataFrame indexed by date with:
    Open, High, Low, Close, Volume
    """
    empty_df = pd.DataFrame(columns=["Open", "High", "Low", "Close", "Volume"])

    for attempt in range(1, max_tries + 1):
        try:
            rows = []

            bars = massive_client.list_aggs(
                ticker=ticker,
                multiplier=1,
                timespan="day",
                from_=str(pd.Timestamp(start_date).date()),
                to=str(pd.Timestamp(end_date).date()),
                adjusted=True,
                sort="asc",
                limit=5000
            )

            for bar in bars:
                rows.append({
                    "Date": pd.to_datetime(bar.timestamp, unit="ms").normalize(),
                    "Open": float(bar.open),
                    "High": float(bar.high),
                    "Low": float(bar.low),
                    "Close": float(bar.close),
                    "Volume": float(bar.volume)
                })

            df = pd.DataFrame(rows)

            if df.empty:
                return empty_df

            df = df.sort_values("Date").drop_duplicates("Date").set_index("Date")
            time.sleep(base_sleep)
            return df

        except Exception as e:
            wait_time = base_sleep * (2 ** (attempt - 1))
            print(f"[daily {ticker}] attempt {attempt}/{max_tries} failed: {e}")

            if attempt < max_tries:
                time.sleep(wait_time)
            else:
                return empty_df

# ---------------------------------------------------------
# Helper: fetch intraday OHLCV using Massive
# 5-minute candles for the entry date
# ---------------------------------------------------------
def fetch_intraday_ohlcv(ticker, target_date, multiplier=5, max_tries=5, base_sleep=2.5):
    """
    Fetch intraday OHLCV bars for one ticker on one date.
    Returns a DataFrame indexed by timezone-adjusted US/Eastern datetime with:
    Open, High, Low, Close, Volume
    """
    empty_df = pd.DataFrame(columns=["Open", "High", "Low", "Close", "Volume"])

    target_date = pd.Timestamp(target_date).normalize()

    for attempt in range(1, max_tries + 1):
        try:
            rows = []

            bars = massive_client.list_aggs(
                ticker=ticker,
                multiplier=multiplier,
                timespan="minute",
                from_=str(target_date.date()),
                to=str(target_date.date()),
                adjusted=True,
                sort="asc",
                limit=50000
            )

            for bar in bars:
                rows.append({
                    "DateTime": pd.to_datetime(bar.timestamp, unit="ms"),
                    "Open": float(bar.open),
                    "High": float(bar.high),
                    "Low": float(bar.low),
                    "Close": float(bar.close),
                    "Volume": float(bar.volume)
                })

            df = pd.DataFrame(rows)

            if df.empty:
                return empty_df

            df = df.sort_values("DateTime").drop_duplicates("DateTime").set_index("DateTime")

            # -------------------------------------------------
            # SAFE TIMEZONE FIX
            # Massive timestamps are typically UTC.
            # Convert intraday display to US/Eastern so chart
            # hours look like actual market hours.
            # -------------------------------------------------
            if df.index.tz is None:
                df.index = df.index.tz_localize("UTC").tz_convert("US/Eastern")
            else:
                df.index = df.index.tz_convert("US/Eastern")

            time.sleep(base_sleep)
            return df

        except Exception as e:
            wait_time = base_sleep * (2 ** (attempt - 1))
            print(f"[intraday {ticker}] attempt {attempt}/{max_tries} failed: {e}")

            if attempt < max_tries:
                time.sleep(wait_time)
            else:
                return empty_df

# ---------------------------------------------------------
# Helper: get trailing daily window
# ---------------------------------------------------------
def get_trailing_window(df, anchor_date, n_days):
    """
    Return last n_days daily rows up to anchor_date.
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=["Open", "High", "Low", "Close", "Volume"])

    d = df[df.index <= pd.Timestamp(anchor_date).normalize()].sort_index()

    if len(d) < n_days:
        return pd.DataFrame(columns=df.columns)

    return d.iloc[-n_days:].copy()

# ---------------------------------------------------------
# Chart style
# Same green/red look
# ---------------------------------------------------------
mc = mpf.make_marketcolors(
    up="green",
    down="red",
    edge="inherit",
    wick="inherit",
    volume="inherit"
)

chart_style = mpf.make_mpf_style(
    base_mpf_style="classic",
    marketcolors=mc,
    facecolor="#d9d9d9",
    gridcolor="#bfbfbf",
    gridstyle="--"
)

# ---------------------------------------------------------
# Helper: save chart safely
# ---------------------------------------------------------
def save_chart(df, save_path, title=None, figsize=(12, 6)):
    """
    Save candlestick chart to disk.
    Safely handles missing title.
    """
    if df is None or df.empty:
        print(f"Skipped empty chart: {save_path}")
        return

    plot_kwargs = {
        "data": df,
        "type": "candle",
        "style": chart_style,
        "figsize": figsize,
        "volume": False,
        "ylabel": "Price ($)",
        "savefig": dict(fname=save_path, dpi=200, bbox_inches="tight")
    }

    # mplfinance does not accept title=None
    if isinstance(title, str):
        plot_kwargs["title"] = title

    mpf.plot(**plot_kwargs)

# ---------------------------------------------------------
# Generate and save charts
# ---------------------------------------------------------
saved_rows = []

for ticker, pseudo in ticker_to_pseudo.items():
    print(f"Generating charts for {ticker} ({pseudo})...")

    time.sleep(2) 

    pseudo_lower = pseudo.lower()

    # Fetch fresh data
    intraday_df = fetch_intraday_ohlcv(
        ticker,
        entry_date,
        multiplier=5,
        max_tries=5,
        base_sleep=2.5
    )

    daily_df = fetch_daily_ohlcv(
        ticker,
        history_start,
        history_end,
        max_tries=5,
        base_sleep=2.5
    )

    # Build daily windows
    week_df = get_trailing_window(daily_df, entry_date, 5)
    month_df = get_trailing_window(daily_df, entry_date, 21)

    # File paths
    intraday_path = os.path.join(output_dir, f"{pseudo_lower}_intraday.png")
    week_path = os.path.join(output_dir, f"{pseudo_lower}_week.png")
    month_path = os.path.join(output_dir, f"{pseudo_lower}_month.png")

    intraday_saved = False
    week_saved = False
    month_saved = False

    # -------------------------
    # Intraday chart
    # -------------------------
    if not intraday_df.empty:
        save_chart(
            intraday_df,
            intraday_path,
            title=None,
            figsize=(14, 8)
        )
        intraday_saved = True
    else:
        print(f"No intraday data for {ticker}; skipped {intraday_path}")

    # -------------------------
    # Week chart
    # -------------------------
    if not week_df.empty:
        save_chart(
            week_df,
            week_path,
            title=None,
            figsize=(14, 8)
        )
        week_saved = True
    else:
        print(f"No week data for {ticker}; skipped {week_path}")

    # -------------------------
    # Month chart
    # -------------------------
    if not month_df.empty:
        save_chart(
            month_df,
            month_path,
            title=None,
            figsize=(14, 8)
        )
        month_saved = True
    else:
        print(f"No month data for {ticker}; skipped {month_path}")

    saved_rows.append({
        "ticker": ticker,
        "pseudoname": pseudo,
        "intraday_saved": intraday_saved,
        "week_saved": week_saved,
        "month_saved": month_saved
    })

# ---------------------------------------------------------
# Summary table
# ---------------------------------------------------------
saved_table = pd.DataFrame(saved_rows)
print("\nSAVED CHART SUMMARY")
display(saved_table)

# ---------------------------------------------------------
# Quick file check
# ---------------------------------------------------------
saved_files = sorted(os.listdir(output_dir))
print(f"\nTotal files in {output_dir}: {len(saved_files)}")
print(saved_files)

Generating charts for NOC (P1)...
Generating charts for LMT (P2)...
Generating charts for JNJ (A1)...
Generating charts for RTX (A2)...
Generating charts for DUK (A3)...
[intraday DUK] attempt 1/5 failed: HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/DUK/range/5/minute/2026-02-19/2026-02-19?adjusted=true&sort=asc&limit=50000 (Caused by ResponseError('too many 429 error responses'))
[intraday DUK] attempt 2/5 failed: HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/DUK/range/5/minute/2026-02-19/2026-02-19?adjusted=true&sort=asc&limit=50000 (Caused by ResponseError('too many 429 error responses'))
[intraday DUK] attempt 3/5 failed: HTTPSConnectionPool(host='api.massive.com', port=443): Max retries exceeded with url: /v2/aggs/ticker/DUK/range/5/minute/2026-02-19/2026-02-19?adjusted=true&sort=asc&limit=50000 (Caused by ResponseError('too many 429 error responses'))
[intraday DUK] at

,ticker,pseudoname,intraday_saved,week_saved,month_saved
0,NOC,P1,True,True,True
1,LMT,P2,True,True,True
2,JNJ,A1,True,True,True
3,RTX,A2,True,True,True
4,DUK,A3,True,True,True
5,CNI,A4,True,True,True
6,PLUG,A5,True,True,True
7,COIN,A6,True,True,True
8,U,A7,True,True,True
9,PM,A8,True,True,True



Total files in assets\images: 48
['a10_intraday.png', 'a10_month.png', 'a10_week.png', 'a1_intraday.png', 'a1_month.png', 'a1_week.png', 'a2_intraday.png', 'a2_month.png', 'a2_week.png', 'a3_intraday.png', 'a3_month.png', 'a3_week.png', 'a4_intraday.png', 'a4_month.png', 'a4_week.png', 'a5_intraday.png', 'a5_month.png', 'a5_week.png', 'a6_intraday.png', 'a6_month.png', 'a6_week.png', 'a7_intraday.png', 'a7_month.png', 'a7_week.png', 'a8_intraday.png', 'a8_month.png', 'a8_week.png', 'a9_intraday.png', 'a9_month.png', 'a9_week.png', 'n1_intraday.png', 'n1_month.png', 'n1_week.png', 'n2_intraday.png', 'n2_month.png', 'n2_week.png', 'n3_intraday.png', 'n3_month.png', 'n3_week.png', 'n4_intraday.png', 'n4_month.png', 'n4_week.png', 'p1_intraday.png', 'p1_month.png', 'p1_week.png', 'p2_intraday.png', 'p2_month.png', 'p2_week.png']
